# Phase 3: 特征工程研究变化

## 概述 Overview

本notebook主要是检查新增Chem分子表征以后的表示效果


**超参数配置：**
- Logistic Regression: L2正则化, balanced权重
- Random Forest: 100棵树, balanced权重
- XGBoost: GPU加速, max_depth=6, balanced权重

**数据配置：**
- 半衰期阈值：SIF：270，SGF：250
- 半衰期异常值：700

**输入**: 
- Phase1_特征工程中的数据内容，被处理好的数据

**输出**：
- Final_Results_xxxxx.json : 30次验证之后的平均结果
- Time_And_Distribute_xxxx.json: 30次验证中的耗时，以及特征的分化情况（如何划分有效特征还是无效特征的）


---

## 1. 环境检查与导入 Environment Setup

In [76]:
# 环境检查
import sys
from pathlib import Path

# 添加项目根目录到路径
project_root = Path.cwd().parent
print(project_root)
sys.path.insert(0, str(project_root / "src"))

# 核心库导入
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
import json
warnings.filterwarnings('ignore')

# 机器学习
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from xgboost import XGBClassifier

# 检查GPU可用性
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        print(f"✓ GPU可用: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠ GPU不可用，将使用CPU训练")
except ImportError:
    gpu_available = False
    print("⚠ PyTorch未安装，将使用CPU训练")

# 设置显示选项
pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ 所有库已成功导入")
print(f"✓ 项目根目录: {project_root}")

d:\RA\feature_extraction
⚠ GPU不可用，将使用CPU训练
✓ 所有库已成功导入
✓ 项目根目录: d:\RA\feature_extraction


In [77]:

def convert_numpy_types(obj):
    """递归转换numpy类型为Python原生类型"""
    import numpy as np
    if isinstance(obj, dict):
        return {k: convert_numpy_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_types(v) for v in obj]
    elif isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj


## 2. 参数配置区 Configuration

**⚙️ 根据您的需求修改以下参数**

In [78]:
# ============== 参数配置区 ==============

CONFIG = {
    # 输入输出路径
    'processed_dir': project_root / 'data' / 'feature-engineering' / 'csv',
    'features_dir': project_root / 'data' / 'feature-engineering' / 'features',
    'cv_results_dir': project_root / 'data' / 'feature-engineering'/ 'outputs' / 'phase3' / 'cv_results',
    'feature_importance_dir': project_root / 'data' / 'feature-engineering' / 'outputs' / 'phase3' / 'feature_importance',
    'transfer_results_dir': project_root / 'data' / 'feature-engineering' / 'outputs' / 'phase3' / 'transfer_results',
    'figures_dir': project_root / 'data' / 'feature-engineering'  / 'figures' / 'phase3',
    'independent_results_dir': project_root / 'data' / 'feature-engineering' / 'outputs' / 'phase3' / 'independent_results',
    # 一个临时输出的文件夹
    'temp':project_root/'data'/'feature-engineering'/'outputs'/'temp',

    # 模型选择（可选: 'lr', 'rf', 'xgb'）
    'models_to_train': ['lr', 'rf', 'xgb'],
    
    # 交叉验证参数
    'n_folds': 5,
    'random_state': 42,
    
    # XGBoost参数
    'use_gpu': gpu_available,
    'xgb_max_depth': 6,
    'xgb_learning_rate': 0.1,
    'xgb_n_estimators': 100,
    
    # Random Forest参数
    'rf_n_estimators': 100,
    'rf_n_jobs': -1,
    
    # Logistic Regression参数
    'lr_max_iter': 1000,
    
    # 可视化参数
    'dpi': 300,
    'format': 'png',
    'display_plots': True,
    'max_display_plots': 8,

    # 阈值
    'threshold':{
        'SIF':270,
        'SGF':250
    },

    # 是否只提取单体分子
    'is_monomer': True,

    # 主要记得修改这里

    # 用于验证模型效果的数据名称
    'dataset_names':{
        #'train':"feature-engineering_train_Morgan(1024)_Avalon(512)_ChemBERTa(384)",
        #'test':"feature-engineering_test_Morgan(1024)_Avalon(512)_ChemBERTa(384)"
    },
    # message,标注信息
    'message':'(0)',# Morgan(1024)_Avalon(512)_ChemBERTa(384)

    # 掩码内容
    'mask':"_error__importance_mask.json"
}

CONFIG['dataset_names']['train']=f"feature-engineering_train_{CONFIG['message']}"
CONFIG['dataset_names']['test']=f"feature-engineering_test_{CONFIG['message']}"

# 创建输出目录
for key in ['cv_results_dir', 'feature_importance_dir', 'transfer_results_dir', 'figures_dir', 'independent_results_dir','temp']:
    CONFIG[key].mkdir(parents=True, exist_ok=True)

print("配置参数:")
print(f"  模型: {CONFIG['models_to_train']}")
print(f"  交叉验证折数: {CONFIG['n_folds']}")
print(f"  GPU加速: {CONFIG['use_gpu']}")
print(f"  XGBoost参数: max_depth={CONFIG['xgb_max_depth']}, lr={CONFIG['xgb_learning_rate']}")

配置参数:
  模型: ['lr', 'rf', 'xgb']
  交叉验证折数: 5
  GPU加速: False
  XGBoost参数: max_depth=6, lr=0.1


## 3. 数据加载与二值化 Data Loading & Binarization

In [79]:
def load_and_binarize_dataset(npz_path: Path, csv_path: Path, target: str, is_monomer: bool = None):
    """
    加载数据并将标签二值化，同时可选择只保留monomer或非monomer样本
    (2025-12-15新增功能)
    
    Args:
        npz_path: NPZ特征文件
        csv_path: 处理后的CSV文件（包含分钟标签）
        target: 'SIF' or 'SGF'
        is_monomer: 如果为True，只保留monomer；False，只保留非monomer；None不筛选
    
    Returns:
        X, y_binary, median_threshold, feature_names
    """
    # 加载NPZ特征
    data = np.load(npz_path, allow_pickle=True)
    X = data['X']
    feature_names = data['feature_names']
    ids_npz = data['ids']
    
    # 加载CSV获取分钟标签
    df = pd.read_csv(csv_path)
    df['id'] = df['id'].astype(str)
    
    # ID匹配：这里是建立一个映射数值
    id_to_idx = {str(id_): idx for idx, id_ in enumerate(ids_npz)}
    valid_indices = [] # 确定当前任务用哪一行
    valid_labels = []
    
    label_col = f"{target}_minutes" # 根据任务目标获取分钟数目
    for _, row in df.iterrows():
        row_id = str(row['id'])
        if row_id in id_to_idx:
            #===================================过滤无效标签=========================================
            label= row[label_col]
            if label == -1 or pd.isna(label):# 判断标签有效（对应的任务就只能做对应的二值化）
                continue
            if is_monomer is not None and row['is_monomer'] != is_monomer:  # 判断是否满足monomer条件
                continue
            if row[f"{target}_minutes"]>700: #特殊情况处理，如果是SIF_minuters>700
                continue   
            #=======================================================================================
            # 符合条件则加入
            valid_indices.append(id_to_idx[row_id]) # 获取对应数据的索引/序号
            valid_labels.append(label)              # 获取label
    
    # 筛选有效样本
    X_valid = X[valid_indices]   # 根据序号直接获取到值，这里是numpy的基本用法之一，比如传入【3，7】那么得到的就是第三行和第七行的数值
    y_minutes = np.array(valid_labels)  # 获取y值

    
    
    # 设定阈值========================
    median =None
    if target=='SIF':
        median = CONFIG['threshold']['SIF']
    elif target=='SGF':
        median = CONFIG['threshold']['SGF']

    # 根据阈值判断是否稳定
    y_binary = (y_minutes >= median).astype(int)  # 1=稳定, 0=不稳定
    
    print(f"  样本数: {len(X_valid)}")
    print(f"  中位数阈值（根据数值分析得到的结果）: {median:.1f} 分钟")
    print(f"  稳定/不稳定: {np.sum(y_binary==1)}/{np.sum(y_binary==0)}")
    
    return X_valid, y_binary, median, feature_names


# 加载所有数据集
datasets_data = {}
npz_files = [
    CONFIG['features_dir']/f"{CONFIG['dataset_names']['train']}.npz",
    CONFIG['features_dir']/f"{CONFIG['dataset_names']['test']}.npz"
]
# 选择加载数据的时候就指定monomer或非monomer样本
is_monomer = CONFIG['is_monomer'] # 仅加载monomer样本，设置为False则加载非monomer样本，None则不筛选

print(f"加载并二值化 {len(npz_files)} 个数据集:\n")
for npz_file in npz_files:
    dataset_name = npz_file.stem.replace('', '')# 这行代码看起来没啥用，其实是历史遗留问题，不用管也不要动
    csv_file = CONFIG['processed_dir'] / f"{dataset_name}.csv"
    
    print(f"{dataset_name}:")

    # 此处新增数据集加载功能（第四个参数）
    
    # SIF
    X_sif, y_sif, median_sif, feat_names = load_and_binarize_dataset(npz_file, csv_file, 'SIF',is_monomer)
    print(f"  SIF数据选择完成，只获得单体数据")
    
    # SGF
    X_sgf, y_sgf, median_sgf, _ = load_and_binarize_dataset(npz_file, csv_file, 'SGF',is_monomer)
    print(f"  SGF数据选择完成，只获取单体数据\n")
    
    datasets_data[dataset_name] = {
        'X_sif': X_sif,
        'y_sif': y_sif,
        'median_sif': median_sif,
        'X_sgf': X_sgf,
        'y_sgf': y_sgf,
        'median_sgf': median_sgf,
        'feature_names': feat_names,
    }

print(f"✓ 数据加载完成！共 {len(datasets_data)} 个数据集")

#=============对数据进行一些处理=========================
# 字典序
print("查看数据集类型",type(datasets_data))   
# dict_keys(['sif_sgf_second', 'US20140294902A1', 'US9624268', 'US9809623B2', 'WO2017011820A2'])
print("尝试输出数据查看情况",datasets_data.keys()) 


#=============上方为数据的处理区域=======================

加载并二值化 2 个数据集:

feature-engineering_train_(0):
  样本数: 391
  中位数阈值（根据数值分析得到的结果）: 270.0 分钟
  稳定/不稳定: 145/246
  SIF数据选择完成，只获得单体数据
  样本数: 296
  中位数阈值（根据数值分析得到的结果）: 250.0 分钟
  稳定/不稳定: 98/198
  SGF数据选择完成，只获取单体数据

feature-engineering_test_(0):
  样本数: 123
  中位数阈值（根据数值分析得到的结果）: 270.0 分钟
  稳定/不稳定: 31/92
  SIF数据选择完成，只获得单体数据
  样本数: 107
  中位数阈值（根据数值分析得到的结果）: 250.0 分钟
  稳定/不稳定: 25/82
  SGF数据选择完成，只获取单体数据

✓ 数据加载完成！共 2 个数据集
查看数据集类型 <class 'dict'>
尝试输出数据查看情况 dict_keys(['feature-engineering_train_(0)', 'feature-engineering_test_(0)'])


## 4. 独立验证训练

In [80]:
import random
# 选择模型：这里已经强制了随机种子1-100000
def get_model(model_name: str, use_gpu: bool = False):
    
    CONFIG['random_state']=random.randint(1, 1000000)
    # print("选择模型为", model_name, "随机种子", CONFIG['random_state'])
    """
    创建模型实例
    """
    if model_name == 'lr':
        return LogisticRegression(
            max_iter=CONFIG['lr_max_iter'],
            class_weight='balanced',
            random_state=random.randint(1, 1000000)
        )
    elif model_name == 'rf':
        return RandomForestClassifier(
            n_estimators=CONFIG['rf_n_estimators'],
            class_weight='balanced',
            n_jobs=CONFIG['rf_n_jobs'],
            random_state=random.randint(1, 1000000)
        )
    elif model_name == 'xgb':
        params = {
            'max_depth': CONFIG['xgb_max_depth'],
            'learning_rate': CONFIG['xgb_learning_rate'],
            'n_estimators': CONFIG['xgb_n_estimators'],
            'random_state': random.randint(1, 1000000),
            'tree_method': 'hist',
        }
        if use_gpu:
            params['device'] = 'cuda:0'
        return XGBClassifier(**params)
    else:
        raise ValueError(f"Unknown model: {model_name}")
# 进行K折交叉验证，
def cross_validate_model(X, y, model_name: str, dataset_name: str, target: str):
    """
    执行k折交叉验证，根据训练数据集自身进行工作
    
    Returns:
        dict: CV结果
    """
    # 自动调整fold数（小数据集）
    min_class_count = np.bincount(y).min()
    n_folds = min(CONFIG['n_folds'], min_class_count)
    if n_folds < CONFIG['n_folds']:
        print(f"    ⚠ 样本数较少，调整fold数为 {n_folds}")
    
    if n_folds < 2:
        print("    ⚠ 样本过少，无法进行分层交叉验证，跳过该任务")
        return None

    
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=CONFIG['random_state'])
    
    metrics = {
        'accuracy': [],
        'precision': [],
        'recall': [],
        'f1': [],
        'auc': []
    }
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        # 训练模型
        model = get_model(model_name, CONFIG['use_gpu'])
        model.fit(X_train, y_train) # 这个地方是训练模型的核心代码
        
        # 预测
        y_pred = model.predict(X_test)
        # （修改后的probo）
        proba = model.predict_proba(X_test)
        if proba.shape[1] == 2:
            y_proba = proba[:, 1]
        else:
            y_proba = None  # 只有一个类别时，AUC 没有定义
        
        # 计算指标
        metrics['accuracy'].append(accuracy_score(y_test, y_pred))
        metrics['precision'].append(precision_score(y_test, y_pred, average='binary', zero_division=0))
        metrics['recall'].append(recall_score(y_test, y_pred, average='binary', zero_division=0))
        metrics['f1'].append(f1_score(y_test, y_pred, average='binary', zero_division=0))
        
        # AUC（需要至少两个类别，新增一种情况与下面配合，就是只有一种类别）
        if y_proba is not None and len(np.unique(y_test)) > 1:
            metrics['auc'].append(roc_auc_score(y_test, y_proba))
        else:
            metrics['auc'].append(np.nan)
    
    # 汇总结果
    results = {
        'dataset': f'cross_{dataset_name}',
        'target': target,
        'model': model_name,
        'n_folds': n_folds,
        'metrics': metrics,
        'mean_metrics': {k: np.nanmean(v) for k, v in metrics.items()},
        'std_metrics': {k: np.nanstd(v) for k, v in metrics.items()},
    }
    
    return results
# 进行独立验证的函数
def independent_validate(X_train, y_train, X_test, y_test, model_name: str, 
                         dataset_train_name: str, 
                         dataset_test_name: str,
                         target: str):
    """
    在训练集上进行全量训练，在独立的测试集上进行单次验证
    
    Returns:
        dict: 评估结果（结构与原CV结果一致，方便兼容）
    """
    # 检查类别情况，防止只有单类别无法计算指标
    unique_classes = np.unique(y_test)
    if len(unique_classes) < 2:
        print(f"    ⚠ 测试集只有类别 {unique_classes}，无法进行二分类评估，跳过该任务")
        return None

    # --- 核心训练过程 ---
    # 0. 刷新数据顺序
    from sklearn.utils import shuffle
    X_train, y_train = shuffle(X_train, y_train, random_state=random.randint(1, 1000000))
    # 1. 创建模型
    model = get_model(model_name, CONFIG['use_gpu'])
    # 2. 直接在全量 X_train 上训练，不再有 Fold 迭代
    model.fit(X_train, y_train) 

    
    
    # --- 预测过程 ---
    y_pred = model.predict(X_test)
    proba = model.predict_proba(X_test)
    
    if proba.shape[1] == 2:
        y_proba = proba[:, 1]
    else:
        y_proba = None

    # --- 计算指标 ---
    # 为了保持输出结构一致，我们将结果存入列表（虽然只有一个值）
    metrics = {
        'accuracy': [accuracy_score(y_test, y_pred)],
        'precision': [precision_score(y_test, y_pred, average='binary', zero_division=0)],
        'recall': [recall_score(y_test, y_pred, average='binary', zero_division=0)],
        'f1': [f1_score(y_test, y_pred, average='binary', zero_division=0)],
        'auc': []
    }
    
    if y_proba is not None and len(unique_classes) > 1:
        metrics['auc'].append(roc_auc_score(y_test, y_proba))
    else:
        metrics['auc'].append(np.nan)

    # --- 汇总结果（保持与原 CV 格式兼容） ---
    results = {
        'dataset': f'{dataset_train_name}_{dataset_test_name}',
        'target': target,
        'model': model_name,
        'fold': 1, #  K-Fold
        'metrics': metrics,
        'mean_metrics': {k: np.nanmean(v) for k, v in metrics.items()}, # 均值即本身
        'std_metrics': {k: 0.0 for k in metrics.keys()},                # 单次验证无标准差
    }
    
    return results, 0




## 4. 进行30次的独立验证，并计算均值。

注意一下执行的是啥东西。。。。

In [81]:
import matplotlib.pyplot as plt
import numpy as np

# 配置
po=1
N_RUNS = 30
METRIC_KEYS = ['accuracy', 'precision', 'recall', 'f1', 'auc']
MODELS = CONFIG['models_to_train']
TARGETS = ['SIF', 'SGF']

# 初始化容器：用于存储每一轮的均值结果
# 结构：storage[验证类型][target_model][metric_name] = [value1, value2, ...]
storage = {
    #'cv': {f"{t}_{m}": {k: [] for k in METRIC_KEYS} for t in TARGETS for m in MODELS},
    'ind': {f"{t}_{m}": {k: [] for k in METRIC_KEYS} for t in TARGETS for m in MODELS}
}

# 计算特征数目/分布情况
# ，以及训练耗时
engine={
    'distribution':{},
    'time':{
        'SIF_lr':[],
        'SIF_rf':[],
        'SIF_xgb':[],
        'SGF_lr':[],
        'SGF_rf':[],
        'SGF_xgb':[],
    }
}
# 首先，在循环开始前加载特征掩码文件
try:
    with open(CONFIG['temp']/CONFIG['mask'], "r") as f:
        mask_data = json.load(f)
    mask_arrays = mask_data.get("mask_arrays", {})
    print("✓ 特征掩码文件加载成功")
except FileNotFoundError:
    print("⚠ 未找到 importance_mask.json 文件，将继续使用所有特征")
    mask_arrays = {}

print(f"🚀 开始大规模验证（共 {N_RUNS} 轮迭代）...")
# --- 独立验证部分 ---
for run in range(1, N_RUNS + 1):
    # 核心：动态修改随机种子，确保每一轮的划分和模型初始化都不同
    train_name=CONFIG['dataset_names']['train']
    test_name=CONFIG['dataset_names']['test']
    train_data = datasets_data.get(train_name)
    test_data = datasets_data.get(test_name)
    if train_data and test_data:
        for target in TARGETS:
            # 寻找importance_mask.json，
            # 里面有一个mask_arrays,是个数组，里面对应的key代表情况，
            # 0代表整个特征需要被放弃，1则代表这个特征需要保留
            X_tr, y_tr = train_data[f'X_{target.lower()}'], train_data[f'y_{target.lower()}']
            X_te, y_te = test_data[f'X_{target.lower()}'], test_data[f'y_{target.lower()}']
            
            for model_name in MODELS:
                task_key = f"{target.upper()}_{model_name.lower()}"  # 例如: "SIF_lr"
                # 设置预备数值
                X_tr_=X_tr
                X_te_=X_te
                
                # 检查是否存在该任务对应的掩码
                if task_key in mask_arrays:
                    mask = mask_arrays[task_key]  # 这是一个由0和1组成的列表
                    
                    # 确保掩码长度与特征数一致
                    if len(mask) == X_tr.shape[1]:
                        # 应用掩码：只保留掩码为1的特征列
                        feature_indices_to_keep = [i for i, val in enumerate(mask) if val == 1]
                        
                        if len(feature_indices_to_keep) > 0:
                            X_tr_ = X_tr[:, feature_indices_to_keep]
                            X_te_ = X_te[:, feature_indices_to_keep]
                            print(f"   应用特征掩码 [{task_key}]：保留 {len(feature_indices_to_keep)}/{len(mask)} 个特征")
                            engine['distribution'][task_key]=[len(feature_indices_to_keep), len(mask)]
                        else:
                            print(f"   警告：掩码 [{task_key}] 过滤后无特征保留，跳过此任务")
                            continue
                    else:
                        print(f"   警告：掩码长度与特征数不符 [{task_key}]， 特征数目为{X_tr.shape[1]}， mask数目为{len(mask)}，跳过掩码过滤")
                else:
                    print(f"   未找到任务 [{task_key}] 的掩码，使用所有特征")
                # --- 新增代码结束 ---

                try:
                    # 注意：这里填入的是修改后的
                    res,secs = independent_validate(X_tr_, y_tr, X_te_, y_te, model_name, train_name, test_name, target)

                    if res:
                        for k in METRIC_KEYS:
                            storage['ind'][f"{target}_{model_name}"][k].append(res['mean_metrics'][k])
                
                except Exception as e: 
                    print(f"  ⚠ 独立验证出错: {e}")
                    continue
    
    if run % 10 == 0:
        print(f"已完成 {run}/{N_RUNS} 轮迭代...")



import json
import numpy as np

output_json_path = CONFIG['temp'] / f"Final_Results_{CONFIG['message']}.json"
final_json_data = {}

print("\n" + "="*30 + " 最终结果汇总 " + "="*30)

for mode in ['ind']:
    print(f"\n[{mode.upper()} 模式最终平均值]:")
    final_json_data[mode] = {}
    for key, metrics in storage[mode].items():
        print(f"  {key}:")
        final_json_data[mode][key] = {}
        for m_name, values in metrics.items():
            if values:
                avg_val = round(float(np.nanmean(values)), 4) 
                print(f"    {m_name}: {avg_val:.4f}")
                final_json_data[mode][key][m_name] = avg_val
            else:
                final_json_data[mode][key][m_name] = None

try:
    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(final_json_data, f, indent=4, ensure_ascii=False)
    print(f"\n[成功] 评估结果已保存至: {output_json_path}")
except Exception as e:
    print(f"\n[失败] 保存 JSON 时出错: {e}")



⚠ 未找到 importance_mask.json 文件，将继续使用所有特征
🚀 开始大规模验证（共 30 轮迭代）...
   未找到任务 [SIF_lr] 的掩码，使用所有特征
   未找到任务 [SIF_rf] 的掩码，使用所有特征
   未找到任务 [SIF_xgb] 的掩码，使用所有特征
   未找到任务 [SGF_lr] 的掩码，使用所有特征
   未找到任务 [SGF_rf] 的掩码，使用所有特征
   未找到任务 [SGF_xgb] 的掩码，使用所有特征
   未找到任务 [SIF_lr] 的掩码，使用所有特征
   未找到任务 [SIF_rf] 的掩码，使用所有特征
   未找到任务 [SIF_xgb] 的掩码，使用所有特征
   未找到任务 [SGF_lr] 的掩码，使用所有特征
   未找到任务 [SGF_rf] 的掩码，使用所有特征
   未找到任务 [SGF_xgb] 的掩码，使用所有特征
   未找到任务 [SIF_lr] 的掩码，使用所有特征
   未找到任务 [SIF_rf] 的掩码，使用所有特征
   未找到任务 [SIF_xgb] 的掩码，使用所有特征
   未找到任务 [SGF_lr] 的掩码，使用所有特征
   未找到任务 [SGF_rf] 的掩码，使用所有特征
   未找到任务 [SGF_xgb] 的掩码，使用所有特征
   未找到任务 [SIF_lr] 的掩码，使用所有特征
   未找到任务 [SIF_rf] 的掩码，使用所有特征
   未找到任务 [SIF_xgb] 的掩码，使用所有特征
   未找到任务 [SGF_lr] 的掩码，使用所有特征
   未找到任务 [SGF_rf] 的掩码，使用所有特征
   未找到任务 [SGF_xgb] 的掩码，使用所有特征
   未找到任务 [SIF_lr] 的掩码，使用所有特征
   未找到任务 [SIF_rf] 的掩码，使用所有特征
   未找到任务 [SIF_xgb] 的掩码，使用所有特征
   未找到任务 [SGF_lr] 的掩码，使用所有特征
   未找到任务 [SGF_rf] 的掩码，使用所有特征
   未找到任务 [SGF_xgb] 的掩码，使用所有特征
   未找到任务 [SIF_lr] 的掩码，使用所有特征
   未找到任务 [SIF_rf] 的掩码，使用所有特征